In [1]:
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
def decide_bit(comp0, comp1, window, method_name):

    left, right = window

    # metric rule
    if method_name in ["compute_mse", "compute_distance_lpips"]:
        reducer = np.min
        better = lambda a, b: a < b
    else:
        reducer = np.max
        better = lambda a, b: a > b

    left = max(0, left)
    right = min(len(comp0), right)

    score0 = reducer(comp0[left:right])
    score1 = reducer(comp1[left:right])

    return int(better(score1, score0))

## Extended analysis - LPIPS

In [8]:
correct_key_ota1 = np.array([0,0,1,1,0])  # fill in the correct key

files = sorted(glob.glob("./results/analysis lpips/analysis_ota1_bitpos*_compute_distance_lpips.npz"))

pattern = re.compile(r"bitpos(?P<bitpos>\d+)_compute_distance_lpips\.npz")
rows = []


for f in files:
    m = pattern.search(f)
    if not m:
        continue

    bit_idx = int(m.group("bitpos"))

    data = np.load(f)

    comp0 = data["comparison_0"]
    comp1 = data["comparison_1"]

    window = (0, len(comp0))  # Use the full range of the comparison arrays

    recovered = decide_bit(comp0, comp1, window, "compute_distance_lpips")

    correct = recovered == correct_key_ota1[bit_idx]
    rows.append({
        "bitpos"   : bit_idx,
        "recovered": recovered,
        "correct" : correct
    })

df_ota1 = pd.DataFrame(rows)



print(df_ota1)
print(f"Recovery rate: {df_ota1['correct'].mean():.2%}")


   bitpos  recovered  correct
0       0          1    False
1       1          1    False
2       2          1     True
3       3          0    False
4       4          1    False
Recovery rate: 20.00%


In [9]:
file_sc10 = np.load("./results/analysis lpips/analysis_ota2_sc10_compute_distance_lpips.npz")
file_sc11 = np.load("./results/analysis lpips/analysis_ota2_sc11_compute_distance_lpips.npz")

sc10_recoveredbit = decide_bit(file_sc10["comparison_0"], file_sc10["comparison_1"], (0, len(file_sc10["comparison_0"])), "compute_distance_lpips")
sc11_recoveredbit = decide_bit(file_sc11["comparison_0"], file_sc11["comparison_1"], (0, len(file_sc11["comparison_0"])), "compute_distance_lpips")


correct = 0

if sc10_recoveredbit == 0:
    correct += 1

if sc11_recoveredbit == 1:
    correct += 1

recovery_rate = correct / 2

print(f"Recovery rate: {recovery_rate:.2%}")


Recovery rate: 0.00%


In [10]:
import glob
import numpy as np

windows = [
    [27870, 47870],
    [55740, 75740],
    [83610, 103610],
    [111480, 131480],
    [139350, 159350],
    [167220, 187220],
]

files = sorted(glob.glob("./results/analysis lpips/analysis_ota3_bitpos*_compute_distance_lpips.npz"))

pattern = re.compile(r"bitpos(?P<bitpos>\d+)_compute_distance_lpips\.npz")
rows = []

true_key = [1,0,1,0,1,0]
for f in files:
    m = pattern.search(f)
    if not m:
        continue

    bit_idx = int(m.group("bitpos"))

    data = np.load(f)

    comp0 = data["comparison_0"]
    comp1 = data["comparison_1"]

    window = windows[bit_idx]

    recovered = decide_bit(comp0, comp1, window, "compute_distance_lpips")

    rows.append({
        "bitpos"   : bit_idx,
        "recovered": recovered
    })

df_ota3 = pd.DataFrame(rows)

df_ota3["correct"] = df_ota3.apply(
    lambda r: r["recovered"] == true_key[r["bitpos"]],
    axis=1
)


print(df_ota3)
print(f"Recovery rate: {df_ota3['correct'].mean():.2%}")


   bitpos  recovered  correct
0       0          1     True
1       1          1    False
2       2          0    False
3       3          1    False
4       4          0    False
5       5          1    False
Recovery rate: 16.67%


## Extended analysis on other similarity metrics

In [27]:
correct_key_ota1 = np.array([0,0,1,1,0])  # fill in the correct key

files = sorted(glob.glob("./results/analysis similarities/ota1_bitpos*_*.npz"))

pattern = re.compile(r"bitpos(?P<bitpos>\d+)_(?P<method>.+)\.npz")
rows = []


for f in files:
    m = pattern.search(f)
    if not m:
        continue

    bit_idx = int(m.group("bitpos"))
    method = m.group("method")

    data = np.load(f)

    comp0 = data["comparison_0"]
    comp1 = data["comparison_1"]

    window = (0, len(comp0))  # Use the full range of the comparison arrays

    recovered = decide_bit(comp0, comp1, window, method)

    correct = recovered == correct_key_ota1[bit_idx]

    rows.append({
        "method"   : method,
        "bitpos"   : bit_idx,
        "recovered": recovered,
        "correct" : correct
    })

df_ota1_sim = pd.DataFrame(rows)
print(df_ota1_sim)
for method in df_ota1_sim["method"].unique():
    method_df = df_ota1_sim[df_ota1_sim["method"] == method]
    recovery_rate = method_df["correct"].mean()
    print(f"Method: {method}, Recovery rate: {recovery_rate:.2%}")





            method  bitpos  recovered  correct
0   compute_cosine       0          0     True
1      compute_mse       0          0     True
2     compute_psnr       0          0     True
3   compute_cosine       1          0     True
4      compute_mse       1          0     True
5     compute_psnr       1          0     True
6   compute_cosine       2          0    False
7      compute_mse       2          1     True
8     compute_psnr       2          1     True
9   compute_cosine       3          0    False
10     compute_mse       3          1     True
11    compute_psnr       3          1     True
12  compute_cosine       4          0     True
13     compute_mse       4          0     True
14    compute_psnr       4          0     True
Method: compute_cosine, Recovery rate: 60.00%
Method: compute_mse, Recovery rate: 100.00%
Method: compute_psnr, Recovery rate: 100.00%


In [25]:
file_sc10 = sorted(glob.glob("./results/analysis similarities/ota2_horizontal_sc10_*.npz"))

pattern = re.compile(
    r"ota2_horizontal_sc10_(?P<method>.+)\.npz"
)

rows = []
for f in file_sc10:
    m = pattern.search(f)
    if not m:
        continue

    method = m.group("method")

    data = np.load(f)

    comp0 = data["comparison_0"]
    comp1 = data["comparison_1"]

    window = (0, len(comp0))  # Use the full range of the comparison arrays

    recovered = decide_bit(comp0, comp1, window, method)

    correct = recovered == 0
    rows.append({
        "method"   : method,
        "recovered": recovered,
        "correct" : correct
    })

df_ota2_sc10 = pd.DataFrame(rows)
print(df_ota2_sc10)



file_sc11 = sorted(glob.glob("./results/analysis similarities/ota2_horizontal_sc11_*.npz"))

pattern = re.compile(
    r"ota2_horizontal_sc11_(?P<method>.+)\.npz"
)

rows = []
for f in file_sc11:
    m = pattern.search(f)
    if not m:
        continue

    method = m.group("method")

    data = np.load(f)

    comp0 = data["comparison_0"]
    comp1 = data["comparison_1"]

    window = (0, len(comp0))  # Use the full range of the comparison arrays

    recovered = decide_bit(comp0, comp1, window, method)

    correct = recovered == 1
    rows.append({
        "method"   : method,
        "recovered": recovered,
        "correct" : correct
    })

df_ota2_sc11 = pd.DataFrame(rows)
print(df_ota2_sc11)



           method  recovered  correct
0  compute_cosine          1    False
1     compute_mse          0     True
2    compute_psnr          0     True
           method  recovered  correct
0  compute_cosine          0    False
1     compute_mse          0    False
2    compute_psnr          0    False


In [28]:
windows = [
    [27870, 47870],
    [55740, 75740],
    [83610, 103610],
    [111480, 131480],
    [139350, 159350],
    [167220, 187220],
]

files = glob.glob("./results/analysis similarities/ota3_bitpos*_*.npz")

pattern = re.compile(
    r"ota3_bitpos(?P<bitpos>\d+)_(?P<method>.+)\.npz"
)

rows = []

true_key = [1,0,1,0,1,0]
for f in files:
    m = pattern.search(f)
    if not m:
        continue

    bit_idx = int(m.group("bitpos"))
    method = m.group("method")

    data = np.load(f)

    comp0 = data["comparison_0"]
    comp1 = data["comparison_1"]

    window = windows[bit_idx]

    recovered = decide_bit(comp0, comp1, window, method)

    correct = recovered == true_key[bit_idx]


    rows.append({
        "method" : method,
        "bitpos"   : bit_idx,
        "recovered": recovered,
        "correct" : correct
    })

df_ota3_analysis_sim = pd.DataFrame(rows)
print(df_ota3_analysis_sim)


for method in df_ota3_analysis_sim["method"].unique():
    method_df = df_ota3_analysis_sim[df_ota3_analysis_sim["method"] == method]
    recovery_rate = method_df["correct"].mean()
    print(f"Method: {method}, Recovery rate: {recovery_rate:.2%}")





            method  bitpos  recovered  correct
0     compute_psnr       5          0     True
1      compute_mse       3          0     True
2   compute_cosine       2          0    False
3   compute_cosine       1          1    False
4      compute_mse       4          0    False
5     compute_psnr       1          0     True
6     compute_psnr       4          0    False
7      compute_mse       5          0     True
8   compute_cosine       4          0    False
9      compute_mse       2          1     True
10    compute_psnr       0          1     True
11  compute_cosine       5          1    False
12    compute_psnr       3          0     True
13     compute_mse       0          1     True
14  compute_cosine       0          1     True
15     compute_mse       1          0     True
16    compute_psnr       2          1     True
17  compute_cosine       3          1    False
Method: compute_psnr, Recovery rate: 83.33%
Method: compute_mse, Recovery rate: 83.33%
Method: compute_cosi